# Description

In this notebook, we benchmark EQL with division algorithm on the set of previously generated SR benchmarks.

In [ ]:
from __future__ import annotations

import csv
import time
import math
from pathlib import Path
from collections import namedtuple
from typing import Dict, Any

import h5py
import numpy as np
import sympy as sp
import tensorflow as tf

from config.benchmark_config import DataCFG, EQLDIV

import src.EQLdiv.EQL_Layer_tf as eql
from src.EQLdiv.data_utils import get_penalty_data
from src.EQLdiv.evaluation import (
    calculate_complexity,
    symbolic_matmul_and_bias,
    symbolic_eql_layer,
    get_symbol_list,
)
from src.EQLdiv.utils import get_div_thresh_fn, step_to_epochs


# ------------------------------------------------------------
# Utilities (IDENTICAL TO KORNS VERSION)
# ------------------------------------------------------------

def expr_has_division(expr_str: str) -> bool:
    e = sp.sympify(expr_str)
    try:
        num, den = sp.fraction(sp.together(e))
        return den != 1
    except Exception:
        # conservative fallback: look for negative powers
        for p in e.atoms(sp.Pow):
            exp = p.exp
            if exp.is_Number:
                try:
                    if float(exp) < 0.0:
                        return True
                except Exception:
                    pass
        return False


def _sum_threshold_penalties() -> tf.Tensor:
    penalties = tf.compat.v1.get_collection("Threshold_penalties")
    if not penalties:
        return tf.constant(0.0, dtype=tf.float32)
    return tf.add_n([tf.reduce_sum(p) for p in penalties])


def _mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))


def _l1_from_model(model):
    terms = []
    for layer in model.eql_layers:
        if hasattr(layer, "_dense") and layer._dense is not None:
            terms.append(tf.reduce_sum(tf.abs(layer._dense.kernel)))
            terms.append(tf.reduce_sum(tf.abs(layer._dense.bias)))
    if not terms:
        return tf.constant(0.0, dtype=tf.float32)
    return tf.add_n(terms)


def _extract_kernels_biases_in_order(model):
    kernels, biases = [], []
    for layer in model.eql_layers:
        kernels.append(layer._dense.kernel.numpy())
        biases.append(layer._dense.bias.numpy())
    return kernels, biases


def _build_dataset(X, y, batch_size, repeats, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(min(10_000, int(X.shape[0])))
    ds = ds.repeat(repeats).batch(batch_size, drop_remainder=False)
    return ds


def _mse_np(yhat, y):
    return float(np.mean((yhat.reshape(-1) - y.reshape(-1)) ** 2))


def _trainable_vars_from_model(model) -> list[tf.Variable]:
    vars_: list[tf.Variable] = []
    for layer in model.eql_layers:
        if not hasattr(layer, "_dense") or layer._dense is None:
            continue
        # kernel and bias are tf.Variables
        vars_.append(layer._dense.kernel)
        vars_.append(layer._dense.bias)
    return vars_


# ------------------------------------------------------------
# Data loading
# ------------------------------------------------------------

def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]

    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float32)
    ytr = g["train"]["y"][...].astype(np.float32).reshape(-1, 1)

    Xti = g["test_interp"]["X"][...].astype(np.float32)
    yti = g["test_interp"]["y"][...].astype(np.float32).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float32)
    yte = g["test_extrap"]["y"][...].astype(np.float32).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


# ------------------------------------------------------------
# EQL-div Model (IDENTICAL INTERFACE)
# ------------------------------------------------------------

class Model(object):

    def __init__(
        self,
        *,
        mode: str,
        metadata: Dict[str, Any],
        layer_width: int,
        num_h_layers: int,
        reg_sched: tuple[float, float],
        output_bound: float,
        weight_init_param: float,
        epoch_factor: int,
        batch_size: int,
        test_div_threshold: float,
        reg_scale: float,
        l0_threshold: float,
        train_val_split: float,
        network_init_seed=None,
        layer_ops=None,
        out_op: str = "reg_div",
    ):
        self.metadata = metadata
        self.train_data_size = int(train_val_split * metadata["train_val_examples"])
        self.width = layer_width
        self.num_h_layers = num_h_layers
        self.weight_init_scale = float(weight_init_param) / math.sqrt(metadata["num_inputs"] + num_h_layers)

        self.reg_start = math.floor(num_h_layers * epoch_factor * reg_sched[0])
        self.reg_end = math.floor(num_h_layers * epoch_factor * reg_sched[1])

        self.output_bound = float(
            output_bound if output_bound is not None else metadata["extracted_output_bound"]
        )
        self.reg_scale = float(reg_scale)
        self.batch_size = batch_size
        self.l0_threshold = float(l0_threshold)
        self.is_training = (mode == "train")

        div_thresh_fn = get_div_thresh_fn(
            self.is_training,
            self.batch_size,
            test_div_threshold,
            train_examples=self.train_data_size,
        )

        reg_div = namedtuple("reg_div", ["repeats", "div_thresh_fn"])

        hidden_kwargs = {op: self.width for op in layer_ops}

        self.eql_layers = [
            eql.EQL_Layer(**hidden_kwargs, weight_init_scale=self.weight_init_scale, seed=network_init_seed)
            for _ in range(self.num_h_layers)
        ]

        if out_op == "reg_div":
            self.eql_layers.append(
                eql.EQL_Layer(
                    reg_div=reg_div(
                        repeats=metadata["num_outputs"],
                        div_thresh_fn=div_thresh_fn,
                    ),
                    weight_init_scale=self.weight_init_scale,
                    seed=network_init_seed,
                )
            )
        elif out_op == "id":
            self.eql_layers.append(
                eql.EQL_Layer(
                    id=metadata["num_outputs"],
                    weight_init_scale=self.weight_init_scale,
                    seed=network_init_seed,
                )
            )
        else:
            raise ValueError(f"Unknown out_op={out_op!r}. Expected 'reg_div' or 'id'.")

    def __call__(self, inputs, global_step):

        num_epochs = step_to_epochs(global_step, self.batch_size, self.train_data_size)

        l1_mask = tf.cast(tf.less(num_epochs, self.reg_end), tf.float32) * tf.cast(
            tf.greater(num_epochs, self.reg_start), tf.float32
        )

        l1_reg_sched = l1_mask * tf.constant(self.reg_scale, dtype=tf.float32)

        l0_threshold = tf.cond(
            tf.less(num_epochs, self.reg_end),
            lambda: tf.constant(0.0, dtype=tf.float32),
            lambda: tf.constant(self.l0_threshold, dtype=tf.float32),
        )

        output = inputs
        for layer in self.eql_layers:
            output = layer(output, l1_reg_sched=l1_reg_sched, l0_threshold=l0_threshold)

        P_bound = (tf.abs(output) - self.output_bound) * tf.cast(
            (tf.abs(output) > self.output_bound), dtype=tf.float32
        )

        return output, P_bound, l1_reg_sched


# ------------------------------------------------------------
# Main experiment
# ------------------------------------------------------------

def main():

    cfg = DataCFG()
    out_csv = Path(EQLDIV.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:

        w = csv.writer(out)
        w.writerow([
            "group",
            "run",
            "seed",
            "out_op",
            "train_mse",
            "test_interp_mse",
            "test_extrap_mse",
            "complexity",
            "duration_s",
            "true_expr",
            "found_expr",
        ])

        for gname in sorted(f.keys()):

            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_train = Xtr.shape[0]
            n_inputs = Xtr.shape[1]

            # penalty bounds
            xmins = Xtr.min(axis=0)
            xmaxs = Xtr.max(axis=0)
            penalty_bounds = [(float(lo), float(hi)) for lo, hi in zip(xmins, xmaxs)]

            y_abs = np.abs(ytr.reshape(-1))
            extracted_output_bound = float(max(1.0, np.quantile(y_abs, 0.99) * 2.0))

            metadata = {
                "train_val_examples": n_train,
                "num_inputs": n_inputs,
                "num_outputs": 1,
                "extracted_output_bound": extracted_output_bound,
                "extracted_penalty_bounds": penalty_bounds,
            }

            n_pen = min(EQLDIV.penalty_examples_cap, n_train)
            X_pen, y_pen = get_penalty_data(
                num_examples=n_pen,
                penalty_bounds=penalty_bounds,
                num_inputs=n_inputs,
                num_outputs=1,
            )

            X_pen = np.asarray(X_pen, dtype=np.float32)
            y_pen = np.asarray(y_pen, dtype=np.float32).reshape(-1, 1)

            # choose out_op based on GT
            out_op = "reg_div" if expr_has_division(true_expr_str) else "id"

            for run in range(EQLDIV.n_runs):

                seed = EQLDIV.base_seed + run
                tf.random.set_seed(seed)
                np.random.seed(seed)

                model = Model(
                    mode="train",
                    metadata=metadata,
                    layer_width=EQLDIV.layer_width,
                    num_h_layers=EQLDIV.num_h_layers,
                    reg_sched=EQLDIV.reg_sched,
                    output_bound=EQLDIV.output_bound,
                    weight_init_param=EQLDIV.weight_init_param,
                    epoch_factor=EQLDIV.epoch_factor,
                    batch_size=EQLDIV.batch_size,
                    test_div_threshold=EQLDIV.test_div_threshold,
                    reg_scale=EQLDIV.reg_scale,
                    l0_threshold=EQLDIV.l0_threshold,
                    train_val_split=1.0,
                    network_init_seed=seed,
                    layer_ops=list(EQLDIV.layer_ops),
                    out_op=out_op,
                )

                optimizer = tf.keras.optimizers.Adam(
                    learning_rate=EQLDIV.learning_rate,
                    beta_1=EQLDIV.beta1,
                )

                global_step = tf.Variable(0, dtype=tf.int64, trainable=False)

                def train_step(xb, yb, use_penalty: bool):
                    tf.compat.v1.get_collection_ref("Threshold_penalties").clear()

                    with tf.GradientTape() as tape:
                        preds, P_bound, l1_reg_sched = model(xb, global_step)

                        bound_penalty = tf.reduce_sum(P_bound)
                        P_theta = _sum_threshold_penalties()
                        mse_loss = _mse(yb, preds)
                        l1_loss = l1_reg_sched * _l1_from_model(model)

                        penalty_loss = P_theta + bound_penalty + l1_loss
                        normal_loss = mse_loss + P_theta + l1_loss
                        loss = penalty_loss if use_penalty else normal_loss

                    vars_ = _trainable_vars_from_model(model)
                    grads = tape.gradient(loss, vars_)
                    optimizer.apply_gradients(zip(grads, vars_))
                    global_step.assign_add(1)

                t0 = time.perf_counter()

                for _ in range(EQLDIV.epoch_factor):

                    ds_pen = _build_dataset(X_pen, y_pen, EQLDIV.batch_size, 1, True)
                    for xb, yb in ds_pen:
                        train_step(xb, yb, True)

                    ds_train = _build_dataset(Xtr, ytr, EQLDIV.batch_size, EQLDIV.penalty_every, True)
                    for xb, yb in ds_train:
                        train_step(xb, yb, False)

                dur = time.perf_counter() - t0

                yhat_tr = model(tf.convert_to_tensor(Xtr), global_step)[0].numpy()
                yhat_ti = model(tf.convert_to_tensor(Xti), global_step)[0].numpy()
                yhat_te = model(tf.convert_to_tensor(Xte), global_step)[0].numpy()

                train_mse = _mse_np(yhat_tr, ytr)
                test_interp_mse = _mse_np(yhat_ti, yti)
                test_extrap_mse = _mse_np(yhat_te, yte)

                kernels, biases = _extract_kernels_biases_in_order(model)
                fns_list = [layer.get_fns() for layer in model.eql_layers]

                complexity = calculate_complexity(
                    kernels, biases, fns_list, thresh=EQLDIV.complexity_threshold
                )

                in_nodes = get_symbol_list(n_inputs)
                res = in_nodes
                for kernel, bias, fns in zip(kernels, biases, fns_list):
                    res = symbolic_matmul_and_bias(res, kernel, bias)
                    res = symbolic_eql_layer(res, fns)

                expr = sp.N(res[0], EQLDIV.round_decimals)

                w.writerow([
                    gname,
                    run,
                    seed,
                    out_op,
                    train_mse,
                    test_interp_mse,
                    test_extrap_mse,
                    complexity,
                    dur,
                    true_expr_str,
                    str(expr),
                ])
                out.flush()

                print(
                    f"[{gname}] run={run} seed={seed} out_op={out_op} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} extrap={test_extrap_mse:.3e}"
                )

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

2026-03-03 17:02:59.270722: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-03 17:02:59.270764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-03 17:02:59.272223: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-03 17:02:59.280800: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 17:03:00.196603: W tensorflow/compiler/tf2

In [ ]:
# from __future__ import annotations

# import math
# from pathlib import Path

# import pandas as pd
# import sympy as sp

# from config.benchmark_config import SINDY


# def _safe_latex(expr_str: str) -> str | None:
#     if not isinstance(expr_str, str) or not expr_str.strip():
#         return None
#     try:
#         expr = sp.sympify(expr_str)
#         return sp.latex(expr)
#     except Exception:
#         return None


# def _format_pm(value: float, std: float, sig: int = 1) -> str:
#     if value == 0.0:
#         return r"(0\pm0)\times 10^{0}"

#     exp = int(math.floor(math.log10(abs(value))))
#     scale = 10 ** exp

#     v = round(value / scale, sig)
#     s = round(std / scale, sig)

#     return rf"({v}\pm{s})\times 10^{{{exp}}}"


# def _count_nodes(expr: sp.Expr) -> int:
#     return sum(1 for _ in sp.preorder_traversal(expr))


# def summarize(csv_path: str | Path, k: int = 5) -> None:
#     df = pd.read_csv(csv_path)

#     metrics = [
#         "train_mse",
#         "test_interp_mse",
#         "test_extrap_mse",
#     ]

#     for gname, gdf in df.groupby("group"):
#         print(f"\n{gname}")

#         # Select k best by extrapolation error
#         gdf = gdf.sort_values("test_extrap_mse").iloc[:k]

#         # ---- error metrics ----
#         for m in metrics:
#             vals = gdf[m].astype(float).to_numpy()
#             mean = float(vals.mean())
#             std = float(vals.std(ddof=0))
#             print(f"  {m}: {_format_pm(mean, std)}")

#         # ---- symbolic complexity ----
#         node_counts = []
#         for s in gdf["found_expr"]:
#             if not isinstance(s, str) or not s.strip():
#                 continue
#             try:
#                 expr = sp.sympify(s)
#                 node_counts.append(_count_nodes(expr))
#             except Exception:
#                 pass

#         if node_counts:
#             nc = pd.Series(node_counts, dtype=float)
#             print(
#                 f"  expr_nodes: "
#                 f"{nc.mean():.1f} ± {nc.std(ddof=0):.1f}"
#             )
#         else:
#             print("  expr_nodes: N/A")

#         # ---- best train-fit expression ----
#         best_row = gdf.sort_values("train_mse").iloc[0]
#         latex_expr = _safe_latex(best_row["found_expr"])

#         if latex_expr is not None:
#             print("  best_train_expr_latex:")
#             print(f"    ${latex_expr}$")
#         else:
#             print("  best_train_expr_latex: N/A")


# if __name__ == "__main__":
#     summarize(SINDY.results_path, k=5)